# 03. Baseline Modelling

## 1. Notebook Objective and Modelling Framework

The feature-engineering pipeline produced a chronological, pre-match dataset containing team-strength, form, scheduling and season-context variables for Premier League fixtures.

The purpose of this notebook is to establish a set of honest and interpretable modelling benchmarks before introducing more flexible nonlinear models.

The prediction problem is multiclass classification. For fixture $i$, the target is:

$$
Y_i \in \{H,D,A\},
$$

where:

- $H$ represents a home win;
- $D$ represents a draw;
- $A$ represents an away win.

Rather than predicting only the most likely outcome, the models will estimate a complete probability distribution:

$$
\hat{\mathbf{p}}_i
=
\left(
\hat{p}_{i,H},
\hat{p}_{i,D},
\hat{p}_{i,A}
\right),
$$

subject to:

$$
\hat{p}_{i,H}
+
\hat{p}_{i,D}
+
\hat{p}_{i,A}
=
1.
$$

This probability-based framing is essential because the eventual objective is not merely to classify matches correctly. The project aims to compare model probabilities with bookmaker-implied probabilities and identify whether the model contains information beyond the market.

### Chronological evaluation

Football matches are naturally ordered through time. A model used in practice is trained on past fixtures and applied to future fixtures.

The dataset will therefore be split chronologically rather than randomly.

A random split could allow the training sample to contain matches played after fixtures in the validation or test sets. Even where individual features are pre-match safe, this would produce an unrealistic evaluation because the model would indirectly learn from future football environments.

The planned split is:

- training set: the earliest seasons;
- validation set: the second-most-recent completed season;
- test set: the most-recent completed season.

The validation set will be used for model development and decision-making. The test set will remain untouched until the final evaluation stage.

### Baseline hierarchy

Three initial probability benchmarks will be constructed.

#### Uniform-probability baseline

The simplest benchmark assigns equal probability to every outcome:

$$
\hat{p}_H
=
\hat{p}_D
=
\hat{p}_A
=
\frac{1}{3}.
$$

This model contains no football information and provides a minimum reference point.

#### Historical-frequency baseline

A second benchmark predicts outcomes using their frequencies in the training data:

$$
\hat{p}_k
=
\frac{N_k}{N},
\qquad
k \in \{H,D,A\},
$$

where $N_k$ is the number of training fixtures with outcome $k$ and $N$ is the total number of training fixtures.

This captures the general home advantage and the historical frequency of draws and away wins.

#### Multinomial logistic regression

The first feature-based model will be multinomial logistic regression.

For outcome class $k$, the model assigns a linear score:

$$
z_{i,k}
=
\beta_{0,k}
+
\mathbf{x}_i^\top \boldsymbol{\beta}_k,
$$

where $\mathbf{x}_i$ contains the pre-match predictor variables for fixture $i$.

The class probabilities are obtained through the softmax transformation:

$$
\hat{p}_{i,k}
=
\frac{\exp(z_{i,k})}
{\sum_{j \in \{H,D,A\}} \exp(z_{i,j})}.
$$

Multinomial logistic regression provides a valuable baseline because it is:

- probabilistic;
- interpretable;
- computationally efficient;
- suitable for regularisation;
- capable of showing whether the engineered features add value beyond unconditional outcome frequencies.

### Evaluation metrics

The primary metric will be multiclass log loss:

$$
\operatorname{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
y_{i,k}\log(\hat{p}_{i,k}),
$$

where $y_{i,k}=1$ when fixture $i$ has outcome $k$ and $0$ otherwise.

Log loss rewards models that assign high probability to the observed result and heavily penalises confident incorrect predictions.

The multiclass Brier score will also be calculated:

$$
\operatorname{Brier}
=
\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
\left(
\hat{p}_{i,k}-y_{i,k}
\right)^2.
$$

Accuracy and the confusion matrix will be reported as secondary diagnostics. They are useful for interpretation but do not fully evaluate probability quality.

The central benchmark question for this notebook is:

> Do the engineered pre-match features allow a simple multinomial model to produce better out-of-sample probabilities than naive outcome-frequency forecasts?

## 2. Load and Validate the Processed Modelling Dataset

The modelling dataset created in `02_feature_engineering.ipynb` will now be loaded from the project’s processed-data directory.

The preferred input is:

`data/processed/premier_league_model_data.parquet`

Parquet is used because it preserves numeric, nullable-integer and date-related data types more reliably than CSV. The CSV export will remain available as a fallback if the Parquet file cannot be loaded.

Before modelling begins, the dataset must be checked to confirm that:

- the file exists in the expected project directory;
- the dataset contains at least one fixture;
- column names are unique;
- exact duplicate rows are absent;
- the identifier columns are available;
- the target column is present;
- the target contains only `H`, `D` and `A`;
- fixture dates can be parsed successfully;
- rows remain chronologically ordered within each season;
- the exported dataset contains no infinite numeric values.

The notebook will locate the repository root dynamically rather than relying on the notebook’s current working directory. This prevents paths such as `notebooks/data/processed/` from being created accidentally when the notebook is executed from inside the `notebooks` directory.

At this stage, no rows will be removed and no predictors will be transformed. The objective is only to verify that the exported feature-engineering output can be treated as the fixed input for the modelling pipeline.

A successful validation will establish the following primary objects:

- `model_data`: the complete processed modelling dataset;
- `season_column`: the season identifier;
- `date_column`: the fixture date;
- `home_team_column`: the home-team identifier;
- `away_team_column`: the away-team identifier;
- `target_column`: the full-time match result.

These objects will be used throughout the remainder of the notebook.

In [2]:
# ============================================================
# 2. Load and Validate the Processed Modelling Dataset
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Locate the project root
# ------------------------------------------------------------

def find_project_root(start_path):
    """
    Find the nearest parent directory containing the Git repository.
    """
    start_path = Path(start_path).resolve()

    for directory in [start_path, *start_path.parents]:
        if (directory / ".git").exists():
            return directory

    raise FileNotFoundError(
        "Could not locate the project root containing the .git directory."
    )


project_root = find_project_root(Path.cwd())

processed_data_directory = (
    project_root
    / "data"
    / "processed"
)

parquet_path = (
    processed_data_directory
    / "premier_league_model_data.parquet"
)

csv_path = (
    processed_data_directory
    / "premier_league_model_data.csv"
)


# ------------------------------------------------------------
# Load the modelling dataset
# ------------------------------------------------------------

if parquet_path.exists():
    model_data = pd.read_parquet(parquet_path)
    loaded_file = parquet_path
    loaded_format = "Parquet"

elif csv_path.exists():
    model_data = pd.read_csv(csv_path)
    loaded_file = csv_path
    loaded_format = "CSV"

else:
    raise FileNotFoundError(
        "Could not find the processed modelling dataset.\n\n"
        f"Checked:\n- {parquet_path}\n- {csv_path}"
    )


# ------------------------------------------------------------
# Identify essential columns
# ------------------------------------------------------------

def find_first_existing_column(
    dataframe,
    candidates,
    label,
    required=True,
):
    """
    Return the first candidate column present in the DataFrame.
    """
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    if required:
        raise KeyError(
            f"Could not identify the {label} column. "
            f"Checked: {candidates}"
        )

    return None


season_column = find_first_existing_column(
    model_data,
    ["Season", "season"],
    label="season",
)

date_column = find_first_existing_column(
    model_data,
    ["Date", "date", "MatchDate", "match_date"],
    label="fixture date",
)

home_team_column = find_first_existing_column(
    model_data,
    ["HomeTeam", "home_team", "Home"],
    label="home-team",
)

away_team_column = find_first_existing_column(
    model_data,
    ["AwayTeam", "away_team", "Away"],
    label="away-team",
)

target_column = find_first_existing_column(
    model_data,
    ["FTR", "Result", "result", "FullTimeResult"],
    label="target",
)


# ------------------------------------------------------------
# Basic structural validation
# ------------------------------------------------------------

assert len(model_data) > 0, (
    "The modelling dataset is empty."
)

assert model_data.columns.is_unique, (
    "The modelling dataset contains duplicate column names."
)

exact_duplicate_rows = int(
    model_data.duplicated().sum()
)

assert exact_duplicate_rows == 0, (
    f"{exact_duplicate_rows} exact duplicate rows were detected."
)


# ------------------------------------------------------------
# Parse and validate fixture dates
# ------------------------------------------------------------

model_data[date_column] = pd.to_datetime(
    model_data[date_column],
    errors="coerce",
    dayfirst=True,
)

assert model_data[date_column].notna().all(), (
    "At least one fixture date could not be parsed."
)


# ------------------------------------------------------------
# Validate fixture identifiers
# ------------------------------------------------------------

identifier_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]

assert model_data[identifier_columns].notna().all().all(), (
    "One or more fixture identifier columns contain missing values."
)

assert (
    model_data[home_team_column]
    != model_data[away_team_column]
).all(), "A fixture cannot contain the same home and away team."

duplicate_fixture_count = int(
    model_data.duplicated(
        subset=identifier_columns,
    ).sum()
)

assert duplicate_fixture_count == 0, (
    f"{duplicate_fixture_count} duplicate fixtures were detected."
)


# ------------------------------------------------------------
# Validate the target
# ------------------------------------------------------------

model_data[target_column] = (
    model_data[target_column]
    .astype("string")
    .str.strip()
    .str.upper()
)

valid_target_values = {"H", "D", "A"}

invalid_target_values = sorted(
    set(
        model_data[target_column]
        .dropna()
        .unique()
    )
    - valid_target_values
)

assert not invalid_target_values, (
    "The target contains invalid values: "
    f"{invalid_target_values}"
)

assert model_data[target_column].notna().all(), (
    "The target contains missing values."
)


# ------------------------------------------------------------
# Confirm chronological ordering within seasons
# ------------------------------------------------------------

chronology_check = (
    model_data
    .groupby(
        season_column,
        sort=False,
    )[date_column]
    .apply(lambda dates: dates.is_monotonic_increasing)
)

assert chronology_check.all(), (
    "Fixtures are not chronologically ordered within every season."
)


# ------------------------------------------------------------
# Check numeric columns for infinite values
# ------------------------------------------------------------

numeric_columns = model_data.select_dtypes(
    include=[np.number]
).columns.tolist()

infinite_value_counts = {
    column: int(
        np.isinf(
            pd.to_numeric(
                model_data[column],
                errors="coerce",
            ).astype(float)
        ).sum()
    )
    for column in numeric_columns
}

columns_with_infinite_values = {
    column: count
    for column, count in infinite_value_counts.items()
    if count > 0
}

assert not columns_with_infinite_values, (
    "Infinite values were detected: "
    f"{columns_with_infinite_values}"
)


# ------------------------------------------------------------
# Create compact validation summaries
# ------------------------------------------------------------

target_distribution = (
    model_data[target_column]
    .value_counts()
    .reindex(["H", "D", "A"])
    .rename_axis("Outcome")
    .reset_index(name="Fixtures")
)

target_distribution["Percentage"] = (
    100
    * target_distribution["Fixtures"]
    / len(model_data)
).round(2)

dataset_summary = pd.DataFrame(
    {
        "Metric": [
            "Fixtures",
            "Columns",
            "Seasons",
            "Earliest fixture",
            "Latest fixture",
            "Numeric columns",
            "Columns with missing values",
        ],
        "Value": [
            f"{len(model_data):,}",
            len(model_data.columns),
            model_data[season_column].nunique(),
            model_data[date_column].min().date(),
            model_data[date_column].max().date(),
            len(numeric_columns),
            int(model_data.isna().any().sum()),
        ],
    }
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Processed modelling dataset loaded successfully.")
print(f"Format: {loaded_format}")
print(f"File: {loaded_file}")
print(f"Shape: {model_data.shape}")
print(f"Target column: {target_column}")

display(dataset_summary)
display(target_distribution)
display(model_data.head(10))

Processed modelling dataset loaded successfully.
Format: Parquet
File: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine\data\processed\premier_league_model_data.parquet
Shape: (3800, 75)
Target column: FTR


,Metric,Value
0,Fixtures,"3,800"
1,Columns,75
2,Seasons,10
3,Earliest fixture,2015-08-08
4,Latest fixture,2025-05-25
5,Numeric columns,70
6,Columns with missing values,47


,Outcome,Fixtures,Percentage
0,H,1691,44.5
1,D,886,23.32
2,A,1223,32.18


,Season,Date,HomeTeam,AwayTeam,FTR,HomeEloBefore,AwayEloBefore,HomeRollingPoints5,AwayRollingPoints5,HomeRollingGoalsFor5,...,PositionDifference,GoalDifferenceDifference,HomeTop4Before,HomeTop6Before,HomeTopHalfBefore,HomeBottom3Before,AwayTop4Before,AwayTop6Before,AwayTopHalfBefore,AwayBottom3Before
0,2015-16,2015-08-08,Bournemouth,Aston Villa,A,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2015-16,2015-08-08,Chelsea,Swansea,D,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2015-16,2015-08-08,Everton,Watford,D,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2015-16,2015-08-08,Leicester,Sunderland,H,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2015-16,2015-08-08,Man United,Tottenham,H,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,2015-16,2015-08-08,Norwich,Crystal Palace,A,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,2015-16,2015-08-09,Arsenal,West Ham,A,1500.0,1500.0,NaN,NaN,NaN,...,7,0,0,0,1,0,0,0,0,0
7,2015-16,2015-08-09,Newcastle,Southampton,D,1500.0,1500.0,NaN,NaN,NaN,...,1,0,0,0,0,0,0,0,0,0
8,2015-16,2015-08-09,Stoke,Liverpool,A,1500.0,1500.0,NaN,NaN,NaN,...,-4,0,0,0,0,0,0,0,1,0
9,2015-16,2015-08-10,West Brom,Man City,A,1500.0,1500.0,NaN,NaN,NaN,...,-1,0,0,0,0,0,0,0,0,0


### Results and Interpretation

The processed modelling dataset loaded successfully from the project-level `data/processed` directory.

The validation confirms that:

- the dataset contains fixtures and modelling variables;
- all column names are unique;
- no exact duplicate rows are present;
- fixture identifiers are complete;
- no fixture contains the same home and away team;
- the target column contains only `H`, `D` and `A`;
- fixture dates were parsed successfully;
- fixtures remain chronologically ordered within each season;
- no numeric column contains infinite values.

The target distribution provides the first indication of class imbalance in Premier League outcomes.

Home wins are generally the most common result, reflecting the historical home advantage. Draws and away wins occur less frequently, meaning that accuracy alone would be an incomplete measure of model quality.

A model could achieve a superficially reasonable accuracy by predicting the most common outcome too often while still producing poor probabilities for draws and away wins. For this reason, later evaluation will prioritise multiclass log loss and Brier score.

The successfully loaded `model_data` DataFrame now represents the fixed input for the baseline-modelling pipeline.

No rows have been removed and no feature transformations have yet been applied.

## 3. Define Identifiers, Target and Predictor Columns

Before constructing the training, validation and test sets, the columns in `model_data` must be separated according to their modelling role.

The dataset contains three distinct groups:

1. fixture identifiers;
2. the prediction target;
3. eligible pre-match predictors.

### Fixture Identifiers

The identifier columns describe each match and are retained for chronological splitting, interpretation and prediction output:

- `Season`
- `Date`
- `HomeTeam`
- `AwayTeam`

These columns will not be passed directly into the baseline logistic-regression model.

`Season` and `Date` are required to preserve the temporal structure of the dataset. The team-name columns identify each fixture but are excluded from the initial baseline because representing team identity directly would require an additional categorical-encoding strategy.

Team strength is already represented through quantitative pre-match variables such as Elo ratings, rolling form and league-table state.

### Target Variable

The prediction target is the full-time result:

`FTR`

with possible values:

$$
Y_i \in \{H,D,A\}.
$$

The classes represent:

- `H`: home win;
- `D`: draw;
- `A`: away win.

For consistent probability output, the class order used throughout the project will be:

$$
(H,D,A).
$$

Maintaining a fixed class order is essential because each predicted-probability column must always correspond to the same outcome.

### Predictor Variables

The predictor set consists of the remaining eligible numeric columns in the processed modelling dataset.

These variables describe information available before kickoff, including:

- pre-match Elo ratings;
- general rolling form;
- venue-specific rolling form;
- rest and fixture congestion;
- relative home–away differences;
- season progress;
- reconstructed pre-match league-table state;
- league-position category indicators.

The predictor matrix will be denoted by:

$$
X \in \mathbb{R}^{N \times P},
$$

where:

- $N$ is the number of fixtures;
- $P$ is the number of predictor variables.

The target vector will be denoted by:

$$
\mathbf{y}
=
(y_1,\ldots,y_N).
$$

### Leakage Protection

The predictor set must not contain:

- `FTR`;
- full-time or half-time goals;
- match statistics recorded during the fixture;
- final-season information;
- bookmaker probabilities or odds;
- temporary feature-engineering columns.

The processed dataset was designed to exclude these variables, but this notebook will validate the separation again before modelling.

### Missing Values

Some predictors contain intentional missing values.

For example, league position and league-position category indicators are undefined before the first completed fixture batch of each season.

Later preprocessing will impute missing numeric values using statistics learned from the training set only.

No imputation, scaling or other transformation will be fitted before the chronological split. This prevents information from the validation or test periods from influencing the training pipeline.

This section will create the following objects:

- `identifier_columns`
- `target_column`
- `feature_columns`
- `X`
- `y`

It will also produce a feature summary confirming the number, data type and missingness of the available predictors.

In [3]:
# ============================================================
# 3. Define Identifiers, Target and Predictor Columns
# ============================================================

# ------------------------------------------------------------
# Define identifier and target columns
# ------------------------------------------------------------

identifier_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]

class_order = ["H", "D", "A"]


# ------------------------------------------------------------
# Identify eligible numeric predictor columns
# ------------------------------------------------------------

excluded_columns = set(
    identifier_columns
    + [target_column]
)

feature_columns = [
    column
    for column in model_data.columns
    if (
        column not in excluded_columns
        and pd.api.types.is_numeric_dtype(model_data[column])
    )
]


# ------------------------------------------------------------
# Validate that predictors have been identified
# ------------------------------------------------------------

assert feature_columns, (
    "No numeric predictor columns were identified."
)

assert len(feature_columns) == len(set(feature_columns)), (
    "The predictor list contains duplicate column names."
)

assert target_column not in feature_columns, (
    "The target column has entered the predictor set."
)

assert not set(identifier_columns).intersection(feature_columns), (
    "One or more identifier columns entered the predictor set."
)


# ------------------------------------------------------------
# Check for obvious leakage columns
# ------------------------------------------------------------

blocked_exact_names = {
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HF",
    "AF",
    "HC",
    "AC",
    "HY",
    "AY",
    "HR",
    "AR",
    "HomeGoals",
    "AwayGoals",
    "HomeScore",
    "AwayScore",
    "FullTimeResult",
}

blocked_name_fragments = [
    "bookmaker",
    "market_probability",
    "marketprob",
    "implied_probability",
    "impliedprob",
    "final_position",
    "finalposition",
]

blocked_predictors = [
    column
    for column in feature_columns
    if (
        column in blocked_exact_names
        or any(
            fragment in column.lower()
            for fragment in blocked_name_fragments
        )
    )
]

assert not blocked_predictors, (
    "Potential leakage columns were detected in the predictor set: "
    f"{blocked_predictors}"
)


# ------------------------------------------------------------
# Construct the predictor matrix and target vector
# ------------------------------------------------------------

X = model_data[feature_columns].copy()
y = model_data[target_column].copy()

fixture_metadata = model_data[
    identifier_columns
].copy()


# ------------------------------------------------------------
# Validate shapes and index alignment
# ------------------------------------------------------------

assert len(X) == len(model_data), (
    "The predictor matrix does not contain every fixture."
)

assert len(y) == len(model_data), (
    "The target vector does not contain every fixture."
)

assert X.index.equals(y.index), (
    "The predictor matrix and target vector are not aligned."
)

assert fixture_metadata.index.equals(X.index), (
    "Fixture metadata and predictors are not aligned."
)

assert X.columns.is_unique, (
    "The predictor matrix contains duplicate columns."
)


# ------------------------------------------------------------
# Validate target classes
# ------------------------------------------------------------

observed_classes = set(
    y.dropna().unique()
)

assert observed_classes == set(class_order), (
    "The observed target classes do not match H, D and A. "
    f"Observed classes: {sorted(observed_classes)}"
)


# ------------------------------------------------------------
# Validate predictor values
# ------------------------------------------------------------

non_numeric_predictors = [
    column
    for column in feature_columns
    if not pd.api.types.is_numeric_dtype(X[column])
]

assert not non_numeric_predictors, (
    "Non-numeric predictor columns were detected: "
    f"{non_numeric_predictors}"
)

infinite_predictor_counts = {}

for column in feature_columns:
    numeric_values = pd.to_numeric(
        X[column],
        errors="coerce",
    ).astype(float)

    infinite_predictor_counts[column] = int(
        np.isinf(numeric_values).sum()
    )

predictors_with_infinite_values = {
    column: count
    for column, count in infinite_predictor_counts.items()
    if count > 0
}

assert not predictors_with_infinite_values, (
    "Infinite values were detected in the predictors: "
    f"{predictors_with_infinite_values}"
)


# ------------------------------------------------------------
# Create the feature summary
# ------------------------------------------------------------

feature_summary = pd.DataFrame(
    {
        "Feature": feature_columns,
        "DataType": [
            str(X[column].dtype)
            for column in feature_columns
        ],
        "MissingValues": [
            int(X[column].isna().sum())
            for column in feature_columns
        ],
        "MissingPercentage": [
            round(
                100 * X[column].isna().mean(),
                2,
            )
            for column in feature_columns
        ],
        "UniqueValues": [
            int(X[column].nunique(dropna=True))
            for column in feature_columns
        ],
        "Minimum": [
            X[column].min(skipna=True)
            for column in feature_columns
        ],
        "Maximum": [
            X[column].max(skipna=True)
            for column in feature_columns
        ],
    }
)

features_with_missing_values = (
    feature_summary[
        feature_summary["MissingValues"] > 0
    ]
    .sort_values(
        by=[
            "MissingPercentage",
            "MissingValues",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Identifiers, target and predictors defined successfully.")
print(f"Fixtures: {len(model_data):,}")
print(f"Identifier columns: {len(identifier_columns)}")
print(f"Target column: {target_column}")
print(f"Predictor columns: {len(feature_columns)}")
print(
    "Predictors containing missing values:",
    len(features_with_missing_values),
)
print(f"Class order: {class_order}")

display(feature_summary)

if not features_with_missing_values.empty:
    display(features_with_missing_values)

Identifiers, target and predictors defined successfully.
Fixtures: 3,800
Identifier columns: 4
Target column: FTR
Predictor columns: 70
Predictors containing missing values: 47
Class order: ['H', 'D', 'A']


,Feature,DataType,MissingValues,MissingPercentage,UniqueValues,Minimum,Maximum
0,HomeEloBefore,float64,0,0.00,3775,1321.589147,1852.597673
1,AwayEloBefore,float64,0,0.00,3776,1316.579895,1854.104860
2,HomeRollingPoints5,float64,502,13.21,15,0.000000,15.000000
3,AwayRollingPoints5,float64,498,13.11,15,0.000000,15.000000
4,HomeRollingGoalsFor5,float64,502,13.21,23,0.000000,24.000000
...,...,...,...,...,...,...,...
65,HomeBottom3Before,Int64,24,0.63,2,0.000000,1.000000
66,AwayTop4Before,Int64,24,0.63,2,0.000000,1.000000
67,AwayTop6Before,Int64,24,0.63,2,0.000000,1.000000
68,AwayTopHalfBefore,Int64,24,0.63,2,0.000000,1.000000


,Feature,DataType,MissingValues,MissingPercentage,UniqueValues,Minimum,Maximum
0,VenuePointsFormDifference5,float64,1016,26.74,30,-14.0,15.0
1,VenueGoalsForFormDifference5,float64,1016,26.74,35,-15.0,20.0
2,VenueGoalsAgainstFormDifference5,float64,1016,26.74,33,-16.0,17.0
3,VenueGoalDifferenceFormDifference5,float64,1016,26.74,50,-24.0,28.0
4,VenueWinRateFormDifference5,float64,1016,26.74,19,-1.0,1.0
5,HomeVenueRollingPoints5,float64,1000,26.32,15,0.0,15.0
6,HomeVenueRollingGoalsFor5,float64,1000,26.32,24,0.0,24.0
7,HomeVenueRollingGoalsAgainst5,float64,1000,26.32,23,0.0,22.0
8,HomeVenueRollingGoalDifference5,float64,1000,26.32,39,-21.0,20.0
9,HomeVenueRollingWinRate5,float64,1000,26.32,6,0.0,1.0


### Results and Interpretation

The modelling columns have now been separated into fixture identifiers, the prediction target and eligible numeric predictors.

The identifier columns are:

- `Season`
- `Date`
- `HomeTeam`
- `AwayTeam`

These variables are retained for chronological splitting, fixture tracking and interpretation, but they are not passed directly into the baseline logistic-regression model.

The target variable is:

- `FTR`

with the fixed class order:

$$
(H,D,A).
$$

This ordering will be preserved whenever predicted probabilities are stored or evaluated, ensuring that each probability column always corresponds to the correct match outcome.

The predictor matrix `X` contains the numeric pre-match variables created during feature engineering, while the target vector `y` contains the observed full-time results.

The validation confirms that:

- the target is not included among the predictors;
- no fixture identifier is included among the predictors;
- every predictor is numeric;
- no predictor contains infinite values;
- predictor names are unique;
- `X`, `y` and `fixture_metadata` contain the same fixtures in the same order;
- all three target classes are present;
- no obvious post-match, bookmaker or final-season variables entered the predictor set.

Some predictor columns contain missing values. These are expected for features that require previous match information or a meaningful pre-match league table.

Missing values have not yet been imputed. Imputation must be learned from the training set only after the chronological split, preventing information from the validation or test periods from influencing the preprocessing pipeline.

The objects now available for modelling are:

- `identifier_columns`
- `target_column`
- `feature_columns`
- `X`
- `y`
- `fixture_metadata`

The next stage is to divide the fixtures into chronological training, validation and test periods.

## 4. Chronological Train–Validation–Test Split

The modelling dataset must now be divided into separate training, validation and test periods.

Because football fixtures occur through time, the split must preserve chronology. A random split would allow the model to train on matches played after fixtures contained in the validation or test sets, producing an unrealistic estimate of future performance.

The split will therefore be performed using complete Premier League seasons.

### Split Design

The seasons will first be ordered according to the date of their earliest fixture.

The dataset will then be divided as follows:

- **training set:** all seasons except the two most recent;
- **validation set:** the second-most-recent season;
- **test set:** the most recent season.

With ten seasons of data, this corresponds to:

- eight seasons for training;
- one season for validation;
- one season for testing.

Let the ordered seasons be:

$$
S_1,S_2,\ldots,S_T.
$$

The training data are:

$$
\mathcal{D}_{\text{train}}
=
\bigcup_{t=1}^{T-2}\mathcal{D}_{S_t},
$$

the validation data are:

$$
\mathcal{D}_{\text{validation}}
=
\mathcal{D}_{S_{T-1}},
$$

and the test data are:

$$
\mathcal{D}_{\text{test}}
=
\mathcal{D}_{S_T}.
$$

### Role of Each Dataset

The training set will be used to:

- estimate preprocessing parameters;
- fit baseline models;
- learn model coefficients.

The validation set will be used to:

- compare modelling choices;
- select regularisation settings;
- assess whether a model improves on the naive benchmarks.

The test set must remain untouched during model development. It will provide the final estimate of performance on the most recent unseen season.

### Preprocessing Discipline

All data-dependent preprocessing must be fitted using the training set only.

This includes:

- missing-value imputation;
- feature scaling;
- any later feature selection;
- model fitting.

For example, if the median of feature $j$ is used for imputation, it must be calculated as:

$$
\widetilde{x}_{j,\text{train}}
=
\operatorname{median}
\left(
X_{\text{train},j}
\right),
$$

and then applied unchanged to the validation and test sets.

Calculating preprocessing statistics from the complete dataset would allow information from future seasons to influence the training process.

### Split Validation

The split will be checked to confirm that:

- every fixture belongs to exactly one dataset;
- no season appears in more than one dataset;
- all training fixtures occur before the validation season;
- all validation fixtures occur before the test season;
- the predictor and target indices remain aligned;
- each dataset contains all three outcome classes;
- the number of fixtures is preserved.

This section will create:

- `train_seasons`
- `validation_season`
- `test_season`
- `X_train`
- `X_validation`
- `X_test`
- `y_train`
- `y_validation`
- `y_test`
- corresponding fixture-metadata objects

The resulting split will remain fixed throughout the later modelling notebooks so that every candidate model is evaluated on the same chronological periods.

In [4]:
# ============================================================
# 4. Chronological Train–Validation–Test Split
# ============================================================

# ------------------------------------------------------------
# Order seasons by the date of their earliest fixture
# ------------------------------------------------------------

season_date_summary = (
    model_data
    .groupby(
        season_column,
        as_index=False,
    )
    .agg(
        SeasonStart=(date_column, "min"),
        SeasonEnd=(date_column, "max"),
        Fixtures=(date_column, "size"),
    )
    .sort_values(
        by="SeasonStart",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ordered_seasons = (
    season_date_summary[season_column]
    .tolist()
)

assert len(ordered_seasons) >= 3, (
    "At least three seasons are required for separate "
    "training, validation and test sets."
)


# ------------------------------------------------------------
# Define the fixed chronological split
# ------------------------------------------------------------

train_seasons = ordered_seasons[:-2]
validation_season = ordered_seasons[-2]
test_season = ordered_seasons[-1]

assert train_seasons, (
    "The training set must contain at least one season."
)

assert validation_season not in train_seasons, (
    "The validation season appears in the training seasons."
)

assert test_season not in train_seasons, (
    "The test season appears in the training seasons."
)

assert validation_season != test_season, (
    "The validation and test seasons must be different."
)


# ------------------------------------------------------------
# Create split masks
# ------------------------------------------------------------

train_mask = model_data[season_column].isin(
    train_seasons
)

validation_mask = (
    model_data[season_column] == validation_season
)

test_mask = (
    model_data[season_column] == test_season
)


# ------------------------------------------------------------
# Confirm that every fixture belongs to exactly one split
# ------------------------------------------------------------

split_membership_count = (
    train_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
)

assert (split_membership_count == 1).all(), (
    "Every fixture must belong to exactly one split."
)

assert int(train_mask.sum()) > 0, (
    "The training set is empty."
)

assert int(validation_mask.sum()) > 0, (
    "The validation set is empty."
)

assert int(test_mask.sum()) > 0, (
    "The test set is empty."
)

assert (
    int(train_mask.sum())
    + int(validation_mask.sum())
    + int(test_mask.sum())
    == len(model_data)
), "The split does not preserve every fixture."


# ------------------------------------------------------------
# Construct predictor, target and metadata splits
# ------------------------------------------------------------

X_train = X.loc[train_mask].copy()
X_validation = X.loc[validation_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_validation = y.loc[validation_mask].copy()
y_test = y.loc[test_mask].copy()

metadata_train = fixture_metadata.loc[
    train_mask
].copy()

metadata_validation = fixture_metadata.loc[
    validation_mask
].copy()

metadata_test = fixture_metadata.loc[
    test_mask
].copy()


# ------------------------------------------------------------
# Validate index alignment within each split
# ------------------------------------------------------------

for split_name, split_X, split_y, split_metadata in [
    (
        "training",
        X_train,
        y_train,
        metadata_train,
    ),
    (
        "validation",
        X_validation,
        y_validation,
        metadata_validation,
    ),
    (
        "test",
        X_test,
        y_test,
        metadata_test,
    ),
]:
    assert split_X.index.equals(split_y.index), (
        f"The {split_name} predictors and target are not aligned."
    )

    assert split_X.index.equals(split_metadata.index), (
        f"The {split_name} predictors and metadata are not aligned."
    )

    assert list(split_X.columns) == feature_columns, (
        f"The {split_name} predictor columns changed."
    )


# ------------------------------------------------------------
# Validate that seasons do not overlap across splits
# ------------------------------------------------------------

observed_train_seasons = set(
    metadata_train[season_column].unique()
)

observed_validation_seasons = set(
    metadata_validation[season_column].unique()
)

observed_test_seasons = set(
    metadata_test[season_column].unique()
)

assert observed_train_seasons == set(train_seasons), (
    "The observed training seasons do not match train_seasons."
)

assert observed_validation_seasons == {
    validation_season
}, "The validation set contains an unexpected season."

assert observed_test_seasons == {
    test_season
}, "The test set contains an unexpected season."

assert observed_train_seasons.isdisjoint(
    observed_validation_seasons
), "Training and validation seasons overlap."

assert observed_train_seasons.isdisjoint(
    observed_test_seasons
), "Training and test seasons overlap."

assert observed_validation_seasons.isdisjoint(
    observed_test_seasons
), "Validation and test seasons overlap."


# ------------------------------------------------------------
# Validate chronological separation
# ------------------------------------------------------------

training_end_date = metadata_train[
    date_column
].max()

validation_start_date = metadata_validation[
    date_column
].min()

validation_end_date = metadata_validation[
    date_column
].max()

test_start_date = metadata_test[
    date_column
].min()

assert training_end_date < validation_start_date, (
    "The training period does not end before validation begins."
)

assert validation_end_date < test_start_date, (
    "The validation period does not end before testing begins."
)

for split_name, split_metadata in [
    ("training", metadata_train),
    ("validation", metadata_validation),
    ("test", metadata_test),
]:
    assert split_metadata[date_column].is_monotonic_increasing, (
        f"The {split_name} fixtures are not chronologically ordered."
    )


# ------------------------------------------------------------
# Validate outcome classes in every split
# ------------------------------------------------------------

expected_classes = set(class_order)

for split_name, split_y in [
    ("training", y_train),
    ("validation", y_validation),
    ("test", y_test),
]:
    observed_split_classes = set(
        split_y.unique()
    )

    assert observed_split_classes == expected_classes, (
        f"The {split_name} set does not contain all target classes. "
        f"Observed: {sorted(observed_split_classes)}"
    )


# ------------------------------------------------------------
# Add a split label for later auditing
# ------------------------------------------------------------

split_labels = pd.Series(
    index=model_data.index,
    dtype="string",
    name="DatasetSplit",
)

split_labels.loc[train_mask] = "Train"
split_labels.loc[validation_mask] = "Validation"
split_labels.loc[test_mask] = "Test"

assert split_labels.notna().all(), (
    "At least one fixture has no dataset-split label."
)


# ------------------------------------------------------------
# Create split summary
# ------------------------------------------------------------

split_summary = pd.DataFrame(
    {
        "Split": [
            "Train",
            "Validation",
            "Test",
        ],
        "Seasons": [
            len(train_seasons),
            1,
            1,
        ],
        "SeasonRange": [
            (
                f"{train_seasons[0]} to "
                f"{train_seasons[-1]}"
            ),
            str(validation_season),
            str(test_season),
        ],
        "Fixtures": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "Percentage": [
            round(
                100 * len(X_train) / len(model_data),
                2,
            ),
            round(
                100 * len(X_validation) / len(model_data),
                2,
            ),
            round(
                100 * len(X_test) / len(model_data),
                2,
            ),
        ],
        "StartDate": [
            metadata_train[date_column].min().date(),
            metadata_validation[date_column].min().date(),
            metadata_test[date_column].min().date(),
        ],
        "EndDate": [
            metadata_train[date_column].max().date(),
            metadata_validation[date_column].max().date(),
            metadata_test[date_column].max().date(),
        ],
    }
)


# ------------------------------------------------------------
# Create class-distribution summary by split
# ------------------------------------------------------------

class_distribution_by_split = pd.concat(
    [
        pd.DataFrame(
            {
                "Split": split_name,
                "Outcome": outcome,
                "Fixtures": int(
                    (split_y == outcome).sum()
                ),
                "Percentage": round(
                    100
                    * (split_y == outcome).mean(),
                    2,
                ),
            },
            index=[0],
        )
        for split_name, split_y in [
            ("Train", y_train),
            ("Validation", y_validation),
            ("Test", y_test),
        ]
        for outcome in class_order
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Chronological data split completed successfully.")
print(f"Training seasons: {train_seasons}")
print(f"Validation season: {validation_season}")
print(f"Test season: {test_season}")
print(
    "Split sizes:",
    f"train={len(X_train):,},",
    f"validation={len(X_validation):,},",
    f"test={len(X_test):,}",
)

display(season_date_summary)
display(split_summary)
display(class_distribution_by_split)

Chronological data split completed successfully.
Training seasons: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
Validation season: 2023-24
Test season: 2024-25
Split sizes: train=3,040, validation=380, test=380


,Season,SeasonStart,SeasonEnd,Fixtures
0,2015-16,2015-08-08,2016-05-17,380
1,2016-17,2016-08-13,2017-05-21,380
2,2017-18,2017-08-11,2018-05-13,380
3,2018-19,2018-08-10,2019-05-12,380
4,2019-20,2019-08-09,2020-07-26,380
5,2020-21,2020-09-12,2021-05-23,380
6,2021-22,2021-08-13,2022-05-22,380
7,2022-23,2022-08-05,2023-05-28,380
8,2023-24,2023-08-11,2024-05-19,380
9,2024-25,2024-08-16,2025-05-25,380


,Split,Seasons,SeasonRange,Fixtures,Percentage,StartDate,EndDate
0,Train,8,2015-16 to 2022-23,3040,80.0,2015-08-08,2023-05-28
1,Validation,1,2023-24,380,10.0,2023-08-11,2024-05-19
2,Test,1,2024-25,380,10.0,2024-08-16,2025-05-25


,Split,Outcome,Fixtures,Percentage
0,Train,H,1361,44.77
1,Train,D,711,23.39
2,Train,A,968,31.84
3,Validation,H,175,46.05
4,Validation,D,82,21.58
5,Validation,A,123,32.37
6,Test,H,155,40.79
7,Test,D,93,24.47
8,Test,A,132,34.74


### Results and Interpretation

The chronological train–validation–test split has been completed successfully.

The seasons were ordered using the date of each season’s earliest fixture and then divided into three non-overlapping periods:

- the earliest seasons form the training set;
- the second-most-recent season forms the validation set;
- the most recent season forms the test set.

This design reflects the way the model will be used in practice: information from earlier fixtures is used to predict matches played later in time.

The validation confirms that:

- every fixture belongs to exactly one split;
- no season appears in more than one split;
- the training period ends before the validation period begins;
- the validation period ends before the test period begins;
- predictor, target and metadata indices remain aligned;
- every predictor set contains the same feature columns;
- all three outcomes, `H`, `D` and `A`, appear in every split;
- the total number of fixtures has been preserved;
- fixtures remain chronologically ordered within each split.

The training set will be used to fit preprocessing transformations and estimate model parameters.

The validation set will be used to compare modelling decisions and assess whether a candidate model improves on the initial benchmarks.

The test set will remain untouched during model development. Its purpose is to provide a final estimate of performance on the most recent unseen season.

This separation is especially important for probability modelling. Repeatedly evaluating modelling choices on the test season would gradually leak information about that season into the development process, even if the model were never directly trained on its fixtures.

No imputation or scaling has yet been applied. These transformations will be fitted using `X_train` only and then applied unchanged to `X_validation` and `X_test`.

The fixed chronological split created in this section will be reused throughout the remaining modelling notebooks so that every model is compared using the same historical periods.